[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C45_Privacy_Trustworthy_Course/05_trustworthy_deploy/05_trustworthy_deploy.ipynb)

# 05 · 可信部署（用 numpy 从零实现）

目标：把 **隐私预算分配**、**组合核算**、**合规清单**、**审计报告** 用 numpy/stdlib 实现，跑通一条**端到端隐私管线**，并 `assert` 验证。

路线：预算分配 → 组合核算(串/并行+RDP) → 端到端管线(DP+联邦+遗忘) → 合规清单 gate → 审计(经验 vs 声称 ε) → ✏️ 练习 → 📖 答案 → 🧪 真实数据胶囊。

> 心智模型：前四模块造零件，本模块**组装整机并质检**。隐私是全生命周期属性：分配预算→核算→合规 gate→审计→文档，缺一不可。

## 1 · 隐私预算分配：把总 ε 切给多个任务

一个系统有总预算（如 ε=4），要分给多个任务（训练、统计发布、未来查询）。实现分配 + 余额核算。

**核心纪律**：分配 + 核算 + 叫停（超支拒绝）。

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

class PrivacyBudget:
    '''隐私预算管理器：分配、核算、叫停三管齐下。'''
    def __init__(self, total_eps):
        self.total = total_eps
        self.spent = 0.0
        self.log = []
    def remaining(self):
        return self.total - self.spent
    def spend(self, eps, task):
        '''串行组合：花费累加。超预算则拒绝(叫停)。'''
        if self.spent + eps > self.total + 1e-12:
            raise ValueError(f'预算不足! 剩 {self.remaining():.3f}, 任务 {task!r} 要 {eps}')
        self.spent += eps
        self.log.append((task, eps))
        return self.remaining()

budget = PrivacyBudget(total_eps=4.0)
budget.spend(1.5, 'DP-SGD 训练')
budget.spend(1.0, '发布统计直方图')
print(f'已花 ε={budget.spent}, 剩余 ε={budget.remaining()}')
assert abs(budget.remaining() - 1.5) < 1e-9
# 叫停：再花 2.0 会超预算
try:
    budget.spend(2.0, '又一次训练')
    assert False, '应该被叫停'
except ValueError as e:
    print('✅ 叫停生效:', str(e)[:40], '...')
print('✅ 预算管理：分配+核算+叫停 三管齐下')

## 2 · 组合核算：串行 vs 并行

回顾模块 01：同一份数据上多任务 = **串行**(ε 相加)；不相交数据子集上 = **并行**(取 max)。
聪明的分配用并行省预算。对拍两者，看并行省了多少。

In [ ]:
def serial_cost(eps_list):
    '''串行组合：同一份数据多次查询，ε 相加。'''
    return sum(eps_list)

def parallel_cost(eps_list):
    '''并行组合：不相交子集各查，取 max。'''
    return max(eps_list)

# 一个直方图有 10 个桶，每桶花 ε=0.3
per_bin = [0.3] * 10
serial = serial_cost(per_bin)      # 若当作串行(错误地)
parallel = parallel_cost(per_bin)  # 正确：各桶数据不相交 -> 并行
print(f'直方图 10 桶各 ε=0.3:')
print(f'  当作串行(错): 总 ε={serial}')
print(f'  并行(正确)  : 总 ε={parallel}  (省了 {serial-parallel:.1f}!)')
assert parallel < serial, '并行组合应远省于串行'
assert parallel == 0.3, '不相交桶 -> 总 ε = 单桶 ε'
print('✅ 直方图各桶数据不相交 -> 用并行组合，总预算=单桶而非求和 (省 10x)')

**RDP 紧致核算**（模块 02）：把多次高斯机制用 RDP 累加，比朴素相加紧。复用简化 RDP 会计验证。

In [ ]:
def rdp_compose(noise_multipliers, orders=None):
    '''多个高斯机制的 RDP 组合(简化)：同阶 RDP 相加，再转 (ε,δ)。'''
    if orders is None:
        orders = np.array([2, 4, 8, 16, 32, 64])
    delta = 1e-5
    best = np.inf
    for a in orders:
        rdp_total = sum(a / (2 * s**2) for s in noise_multipliers)  # 高斯 RDP(α)=α/(2σ²)
        eps = rdp_total + np.log(1/delta)/(a-1)
        best = min(best, eps)
    return float(best)

# 5 个高斯机制(σ=2 各)
sigmas = [2.0]*5
rdp_eps = rdp_compose(sigmas)
# 朴素：每个机制单独的 ε 相加(更松)
naive_eps = sum(rdp_compose([s]) for s in sigmas)
print(f'5 个 σ=2 的高斯机制:')
print(f'  RDP 紧致组合: ε={rdp_eps:.3f}')
print(f'  朴素相加    : ε={naive_eps:.3f}')
assert rdp_eps < naive_eps, 'RDP 组合应比朴素相加紧'
print('✅ RDP 紧致核算 < 朴素相加 —— 同隐私下能做更多事')

## 3 · 端到端隐私管线：DP + 联邦 + 遗忘 组装

把四个模块的工具组装成一条管线，每个环节消费预算、记录隐私性质。这是「从工具到系统」的核心。

In [ ]:
def privacy_pipeline(total_eps=4.0):
    '''端到端隐私管线：数据收集->联邦+DP训练->部署->遗忘，全程核算预算。'''
    budget = PrivacyBudget(total_eps)
    report = {}
    # 阶段1：数据收集 —— 联邦学习(数据不出域，不耗 ε，靠架构)
    report['data'] = {'method': 'federated', 'data_leaves_device': False}
    # 阶段2：训练 —— DP-SGD(client-level DP)，花 ε
    budget.spend(2.5, 'DP-SGD 联邦训练')
    report['train'] = {'method': 'DP-SGD + FedAvg + SecAgg', 'eps_spent': 2.5}
    # 阶段3：部署 —— 留预算给上线后的统计发布
    budget.spend(1.0, '上线统计发布')
    report['deploy'] = {'eps_spent': 1.0}
    # 阶段4：运营 —— 遗忘通道(响应删除请求，不额外耗训练 ε)
    report['ops'] = {'unlearning': 'SISA + influence', 'rtbf_supported': True}
    report['budget'] = {'total': total_eps, 'spent': budget.spent, 'remaining': budget.remaining()}
    return report, budget

report, budget = privacy_pipeline()
import json
print(json.dumps(report, indent=2, ensure_ascii=False))
assert report['data']['data_leaves_device'] == False, '联邦: 数据不出域'
assert report['ops']['rtbf_supported'] == True, '应支持被遗忘权'
assert budget.remaining() >= 0, '预算不应超支'
print(f"\n✅ 端到端管线: 联邦(数据不出域)+DP(ε={report['train']['eps_spent']})+遗忘, 预算余 {budget.remaining():.1f}")

## 4 · 合规清单：上线前必须全绿的 gate

把 GDPR 义务翻译成**可执行的检查函数**。每条返回通过/不通过，组成上线 gate（任一不通过则拦截）。

这就是「合规即代码」——合规从拍脑袋变成 CI 里自动跑的检查。

In [ ]:
def compliance_checklist(system):
    '''GDPR 合规清单：每条义务一个检查，返回 {检查项: 通过bool}。'''
    checks = {}
    checks['知情同意 consent']       = system.get('all_data_consented', False)
    checks['数据最小化 minimization'] = system.get('uses_federation', False) or system.get('minimal_data', False)
    checks['被遗忘权 RTBF']          = system.get('unlearning_available', False)
    checks['量化隐私保证']           = system.get('dp_epsilon', None) is not None and system['dp_epsilon'] < 10
    checks['预算未超支']            = system.get('eps_spent', 1e9) <= system.get('eps_budget', 0)
    checks['文档齐全 model card']    = system.get('has_model_card', False)
    return checks

def gate(checks):
    '''上线 gate：全绿才放行。'''
    return all(checks.values())

# 一个合规的系统
good_system = dict(all_data_consented=True, uses_federation=True, unlearning_available=True,
                   dp_epsilon=2.5, eps_spent=3.5, eps_budget=4.0, has_model_card=True)
checks = compliance_checklist(good_system)
for k, v in checks.items():
    print(f'  [{"✅" if v else "❌"}] {k}')
assert gate(checks), '合规系统应通过 gate'
print(f'\n上线 gate: {"✅ 放行" if gate(checks) else "❌ 拦截"}')

**反例**：一个漏了合规项的系统应被 gate 拦截。验证 gate 真的能拦。

In [ ]:
# 一个有问题的系统：没有遗忘通道、ε 过大、没文档
bad_system = dict(all_data_consented=True, uses_federation=True, unlearning_available=False,
                  dp_epsilon=50.0, eps_spent=3.0, eps_budget=4.0, has_model_card=False)
bad_checks = compliance_checklist(bad_system)
failed = [k for k, v in bad_checks.items() if not v]
print('未通过的检查项:', failed)
assert not gate(bad_checks), '有问题的系统应被 gate 拦截'
assert '被遗忘权 RTBF' in failed and '文档齐全 model card' in failed
print('✅ gate 拦截了不合规系统(缺遗忘通道/ε过大/没文档) —— 合规即代码')

## 5 · 审计：经验隐私 vs 声称隐私

用攻击验证防御（呼应 C12/C44）。模拟成员推断审计：声称的 ε 与经验测得的隐私是否相符？

用金丝雀（canary）思路：注入可识别样本，看模型记住它的程度（成员推断成功率），反推经验隐私。

In [ ]:
def empirical_membership_auc(member_losses, nonmember_losses):
    '''成员推断的 AUC(Mann-Whitney)：随机抽一对(成员,非成员)，成员损失更低的概率。
       AUC=0.5 -> 无法区分(隐私好)；AUC->1 -> 完全可区分(泄露)。'''
    m = np.asarray(member_losses); nm = np.asarray(nonmember_losses)
    wins = sum((m[:, None] < nm[None, :]).sum() for _ in [0]) / (len(m)*len(nm))
    return float(wins)

# 模拟两个系统的审计：强隐私(小ε,噪声大) vs 弱隐私(大ε,噪声小)
def simulate_losses(privacy_noise, n=500, rng=None):
    '''噪声越大(强隐私) -> 成员/非成员损失越难区分。'''
    rng = rng or np.random.default_rng(0)
    member = rng.normal(1.0, privacy_noise, n)      # 成员损失略低
    nonmember = rng.normal(1.5, privacy_noise, n)   # 非成员损失略高
    return member, nonmember

# 强隐私：大噪声 -> AUC 接近 0.5
m_strong, nm_strong = simulate_losses(privacy_noise=2.0, rng=rng)
auc_strong = empirical_membership_auc(m_strong, nm_strong)
# 弱隐私：小噪声 -> AUC 接近 1
m_weak, nm_weak = simulate_losses(privacy_noise=0.2, rng=rng)
auc_weak = empirical_membership_auc(m_weak, nm_weak)
print(f'强隐私系统(大噪声): 成员推断 AUC = {auc_strong:.3f}  (接近0.5 -> 难区分 -> 隐私好)')
print(f'弱隐私系统(小噪声): 成员推断 AUC = {auc_weak:.3f}  (接近1.0 -> 易区分 -> 泄露)')
assert auc_strong < auc_weak, '强隐私应有更低的成员推断 AUC'
assert auc_strong < 0.65, '强隐私应接近不可区分(0.5)'
print('✅ 审计：用成员推断 AUC 经验度量实际隐私 —— 检验声称的 ε 是否名副其实')

---
## ✏️ 练习 1：隐私预算分配

实现 `allocate_budget(total_eps, weights)`：按权重 `weights`（字典 {任务名: 权重}）把总预算 `total_eps` 按比例分给各任务，返回 {任务名: 分得的 ε}。验证分得的 ε 之和 = 总预算。

In [ ]:
def allocate_budget(total_eps, weights):
    # TODO: 按 weights 的比例把 total_eps 分给各任务，返回 {任务: ε}
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
alloc = allocate_budget(4.0, {'训练': 3, '统计': 1})
print('分配结果:', {k: round(v,2) for k,v in alloc.items()})
assert abs(sum(alloc.values()) - 4.0) < 1e-9, '分得的 ε 之和应=总预算'
assert abs(alloc['训练'] - 3.0) < 1e-9 and abs(alloc['统计'] - 1.0) < 1e-9, '应按 3:1 分配'
print('✅ 练习 1 通过：按权重分配隐私预算')

## ✏️ 练习 2：组合核算（混合串行+并行）

一个系统在**同一份数据**上做 `k` 个任务（串行），但其中有些任务是在**不相交子集**上的并行任务组。

实现 `total_epsilon(serial_tasks, parallel_groups)`：`serial_tasks` 是串行任务的 ε 列表，`parallel_groups` 是若干并行组（每组是 ε 列表，组内取 max）。返回总 ε（串行部分相加 + 各并行组的 max 再相加）。

In [ ]:
def total_epsilon(serial_tasks, parallel_groups):
    # TODO: 串行任务 ε 直接相加；每个并行组取 max；全部相加为总 ε
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
# 2 个串行任务(0.5, 0.5) + 1 个并行组[0.3,0.3,0.3](取max=0.3)
total = total_epsilon([0.5, 0.5], [[0.3, 0.3, 0.3]])
print(f'总 ε = {total}')
assert abs(total - 1.3) < 1e-9, '0.5+0.5 + max(0.3,0.3,0.3)=1.3'
# 多个并行组
total2 = total_epsilon([1.0], [[0.2,0.2], [0.4,0.1]])
assert abs(total2 - (1.0 + 0.2 + 0.4)) < 1e-9, '1.0 + max(.2,.2) + max(.4,.1)'
print('✅ 练习 2 通过：会做串行+并行混合的组合核算')

## ✏️ 练习 3：合规清单 gate

实现 `audit_report(system, claimed_eps, empirical_auc)`：生成审计报告字典，包含 `claimed_eps`、`empirical_auc`、以及 `consistent`（经验是否与声称相符——若声称强隐私 claimed_eps<1 但 empirical_auc>0.7，则不一致）。

In [ ]:
def audit_report(system, claimed_eps, empirical_auc):
    # TODO: 返回 dict(claimed_eps=..., empirical_auc=..., consistent=bool)
    #       不一致的判据：claimed_eps < 1.0(声称强隐私) 但 empirical_auc > 0.7(实际易区分)
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
# 案例A：声称强隐私(ε=0.5)，审计 AUC=0.55(确实难区分) -> 一致
rep_a = audit_report({}, claimed_eps=0.5, empirical_auc=0.55)
assert rep_a['consistent'] == True, '声称强隐私且审计也难区分 -> 一致'
# 案例B：声称强隐私(ε=0.5)，但审计 AUC=0.9(实际易区分) -> 不一致(实现可能有bug!)
rep_b = audit_report({}, claimed_eps=0.5, empirical_auc=0.9)
assert rep_b['consistent'] == False, '声称强隐私但审计易区分 -> 不一致(警报)'
print('案例A(声称强/实际强):', rep_a['consistent'], '| 案例B(声称强/实际弱):', rep_b['consistent'])
print('✅ 练习 3 通过：审计能发现「声称 vs 实际」隐私不符(可能是实现 bug)')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def allocate_budget(total_eps, weights):
    s = sum(weights.values())
    return {k: total_eps * w / s for k, w in weights.items()}

In [ ]:
# 练习 2 参考答案
def total_epsilon(serial_tasks, parallel_groups):
    return sum(serial_tasks) + sum(max(g) for g in parallel_groups)

In [ ]:
# 练习 3 参考答案
def audit_report(system, claimed_eps, empirical_auc):
    inconsistent = (claimed_eps < 1.0) and (empirical_auc > 0.7)
    return dict(claimed_eps=claimed_eps, empirical_auc=empirical_auc,
                consistent=not inconsistent)

---
## 🧪 真实数据胶囊：美国 2020 人口普查的预算分配

美国 2020 人口普查是 DP 最大规模的真实部署：一个全局隐私预算，要分配给从国家到街区的各级、各类统计发布。分配方案直接决定哪些数据更准、哪些更糊——这是一场技术+政策的真实博弈。

我们用人口普查风格的**真实结构**（多级地理、按重要性分配 ε）做预算分配，并验证「分得 ε 多 -> 误差小」这条权衡。

（结构与量级参照公开资料；本环境用合成数据，关注预算分配机制。）

In [ ]:
# 人口普查风格：总预算分给各级地理统计(量级示意)
CENSUS_LEVELS = {
    'US (国家)':    4.0,    # 权重最高 -> 最准
    'State (州)':   2.0,
    'County (县)':  1.0,
    'Tract (普查区)': 0.5,
    'Block (街区)': 0.25,   # 最细粒度 -> 最糊
}
TOTAL_EPS = 8.0
census_alloc = allocate_budget(TOTAL_EPS, CENSUS_LEVELS)
print(f"{'地理级别':<16s} {'分得ε':>8s} {'相对误差':>10s}")
for level in CENSUS_LEVELS:
    eps = census_alloc[level]
    rel_error = 1.0 / eps          # 误差 ∝ 1/ε (噪声 ∝ 1/ε)
    print(f'{level:<16s} {eps:>8.3f} {rel_error:>10.3f}')
assert abs(sum(census_alloc.values()) - TOTAL_EPS) < 1e-9, '总预算守恒'
# 分得 ε 越多 -> 误差越小
assert census_alloc['US (国家)'] > census_alloc['Block (街区)'], '国家级分得 ε 多于街区级'
assert (1/census_alloc['US (国家)']) < (1/census_alloc['Block (街区)']), '分得多->误差小'
print('\n✅ 人口普查式预算分配：国家级 ε 多(更准)，街区级 ε 少(更糊) —— 分配=把准确性分给谁')

**🧪 胶囊练习**：实现 `accuracy_under_budget(alloc)`：给定分配 `{级别: ε}`，返回 `{级别: 相对准确性}`（用 `ε/(ε+1)` 当准确性代理，单调递增），并验证「分得 ε 越多，准确性越高」。

In [ ]:
def accuracy_under_budget(alloc):
    # TODO: 返回 {级别: eps/(eps+1)}（ε 越大准确性越高的代理）
    raise NotImplementedError

In [ ]:
# 自测
acc = accuracy_under_budget(census_alloc)
print('各级准确性代理:', {k: round(v,3) for k,v in acc.items()})
assert acc['US (国家)'] > acc['Block (街区)'], '分得 ε 多 -> 准确性高'
assert all(0 < v < 1 for v in acc.values()), '准确性代理应在(0,1)'
print('✅ 胶囊练习通过：量化了预算分配对各级准确性的影响')

In [ ]:
# 📖 胶囊参考答案
def accuracy_under_budget(alloc):
    return {k: eps/(eps+1) for k, eps in alloc.items()}

### 小结
- **从工具到系统**：DP+联邦+安全聚合+遗忘**正交叠加**，各防一环、各补缺口；匹配威胁而非堆砌技术。
- **隐私预算管理**：分配+核算+叫停三管齐下；并行组合省预算、RDP 会计更紧；「预算无限」是危险错觉。
- **合规即代码**：把 GDPR 义务(同意/最小化/RTBF/可问责)翻译成可执行检查 gate，上线前必须全绿。
- **端侧部署**：架构层隐私(数据不上传)，与联邦互补，有自己的缺口(模型下发/难审计)。
- **审计**：用攻击(成员推断/金丝雀)验证「声称 vs 实际」的隐私；实现 bug 比理论缺陷更常见。
- **文档与问责**：模型卡/数据说明书让隐私可检验；可信 ML = 把隐私当全生命周期工程管理。

**🎓 全课完结**：你已从 DP 的数学定义、DP-SGD 的训练改造、联邦的去中心化、遗忘的事后撤回，走到了端到端可信部署。隐私不是一句口号，而是可证明(DP)、可度量(审计)、可审计(文档)的工程。去把它写进真实系统吧。